# HS4002 Week 8

## Practising with Count Models

Research question: **what factors influence the number of children someone has?**

We'll compare three model types for count outcomes:
1. OLS (baseline)
2. Poisson regression
3. Negative Binomial regression

Plus: overdispersion testing and marginal effects.

In [1]:
# !pip install pandas numpy plotnine statsmodels stargazer marginaleffects scipy

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [2]:
raw_df = pd.read_csv("gss2022_mini2.csv")

chosen_variables = [
	'id', 'age', 'happy', 'yrlvmus', 'yrartxbt', 'yrmovie', 'yrcreat',
	'income16', 'prestg10', 'degree', 'race', 'sex', 'wrkstat', 'biokids', 'educ'
]
df = raw_df[chosen_variables].dropna().copy()

income_map = {
	1: 500,    2: 2000,   3: 3500,   4: 4500,   5: 5500,   6: 6500,
	7: 7500,   8: 9000,   9: 11250,  10: 13750, 11: 16250, 12: 18750,
	13: 21250, 14: 23750, 15: 27500, 16: 32500, 17: 37500, 18: 45000,
	19: 55000, 20: 67500, 21: 82500, 22: 100000, 23: 120000, 24: 140000,
	25: 160000, 26: 250000
}
df['income_cont']   = df['income16'].map(income_map)
df['income_10k']   = df['income_cont'] / 10000
df['age']       = raw_df.loc[df.index, 'age'] - 18

df['binary_lvmus']  = (df['yrlvmus']  == 1).astype(int)
df['binary_artxbt'] = (df['yrartxbt'] == 1).astype(int)
df['binary_movie']  = (df['yrmovie']  == 1).astype(int)
df['binary_creat']  = (df['yrcreat']  == 1).astype(int)
df['omni'] = df[['binary_lvmus','binary_artxbt','binary_movie','binary_creat']].sum(axis=1)

df['race_bin']   = pd.Categorical(df['race'].astype(str))
df['race_cat']   = df['race'].map({1: 'White', 2: 'Black'}).fillna('Other')
df['ba_binary']  = (df['degree'] >= 3).astype(int)
df['sex_woman']  = pd.Categorical(
    np.where(df['sex'] == 2, 'Woman', 'Not Woman'),
    categories=['Not Woman', 'Woman']
)
df['working']    = np.where(df['wrkstat'] <= 3, 'Working', 'Not Working')
df['happy_bin']  = (df['happy'] <= 2).astype(int)
df['happy_cat']  = np.where(df['happy'] <= 2, 'Happy', 'Not Happy')

# Starting with an OLS model

Regress no. of children (`biokids`) against income in ten thousands (`income_10k`). Note: one unit = $10,000. Name this `ols_lm1`.

In [3]:
ols_lm1 = smf.ols('biokids ~ income_10k', data=df).fit()
print(ols_lm1.summary())

                            OLS Regression Results                            
Dep. Variable:                biokids   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.002
Method:                 Least Squares   F-statistic:                   0.04228
Date:                Thu, 12 Nov 2026   Prob (F-statistic):              0.837
Time:                        01:06:25   Log-Likelihood:                -1050.5
No. Observations:                 557   AIC:                             2105.
Df Residuals:                     555   BIC:                             2114.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.5647      0.107     14.639      0.0

How would you interpret the coefficient for `income_10k`? What is unusual about using OLS on a count variable?
an increase in income by 10k is associated with an increase of 0.0020 children.
unusual to use OLS as you can't have 0.0020 children

# Continuing with a Poisson Model

Use `smf.poisson()` — equivalent to `glm(..., family='poisson')` in R.

In [ ]:
# poisson model assumes that most people have the same number of kids
po_lm1 = smf.poisson('biokids ~ income_10k', data=df).fit()
print(po_lm1.summary())

Optimization terminated successfully.
         Current function value: 1.767601
         Iterations 3
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      555
Method:                           MLE   Df Model:                            1
Date:                Thu, 12 Nov 2026   Pseudo R-squ.:               3.454e-05
Time:                        01:06:54   Log-Likelihood:                -984.55
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                    0.7942
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4478      0.053      8.410      0.000       0.343       0.552
income_10k     0.0012      0.

Interpretation: a 10k increase in income is expected to cause a 0.0012 increase in the log count of number of children, holding all else constant

In [ ]:
import numpy as np
b = po_lm1.params['income_10k']
print(f'Factor of change, exp(b): {np.exp(b):.4f}')
print(f'Percentage change, 100 * [exp(b) - 1]: {100 * (np.exp(b) - 1):.2f}%')

Factor of change, exp(b): 1.0012
Percentage change, 100 * [exp(b) - 1]: 0.12%


# Adding Age

Add `age` to both models. `age` is centered at 18, so 0 is an 18-year-old. Save as `ols_lm2` and `po_lm2`.

In [6]:
ols_lm2 = smf.ols('biokids ~ income_10k + age', data=df).fit()
po_lm2  = smf.poisson('biokids ~ income_10k + age', data=df).fit()
print(po_lm2.summary())

Optimization terminated successfully.
         Current function value: 1.720780
         Iterations 5
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Thu, 12 Nov 2026   Pseudo R-squ.:                 0.02652
Time:                        01:11:04   Log-Likelihood:                -958.47
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                 4.563e-12
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0674      0.094     -0.720      0.472      -0.251       0.116
income_10k     0.0008      0.

## Comparing the Poisson Models via Likelihood Ratio Test

In [ ]:
lr_stat = 2 * (po_lm2.llf - po_lm1.llf)
df_diff = po_lm2.df_model - po_lm1.df_model
p_val   = chi2.sf(lr_stat, df_diff)
print(f'LR statistic = {lr_stat:.4f}, df = {df_diff:.0f}, p-value = {p_val:.6f}')
# since p-value is < 0.01, we reject the null hypothesis. adding the variabl age made the model fit better

LR statistic = 52.1579, df = 1, p-value = 0.000000


# Building a Negative Binomial Model

Negative binomial relaxes the Poisson assumption that variance = mean (allowing overdispersion).

Equivalent to R's `MASS::glm.nb()`.

Read the coefficients the same way as in a Poisson model: as a factor of change, exp(β), or as a percentage change, 100 × [exp(β) − 1]%.

In [ ]:
# negative binomial model allows for messier counts --> people have variosu number of kids and values are spread out

nb_m1 = smf.negativebinomial('biokids ~ income_10k + age', data=df).fit()
print(nb_m1.summary())

Optimization terminated successfully.
         Current function value: 1.670251
         Iterations: 14
         Function evaluations: 19
         Gradient evaluations: 19
                     NegativeBinomial Regression Results                      
Dep. Variable:                biokids   No. Observations:                  557
Model:               NegativeBinomial   Df Residuals:                      554
Method:                           MLE   Df Model:                            2
Date:                Thu, 12 Nov 2026   Pseudo R-squ.:                 0.01718
Time:                        01:20:09   Log-Likelihood:                -930.33
converged:                       True   LL-Null:                       -946.59
Covariance Type:            nonrobust   LLR p-value:                 8.681e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0832      0.116     -

interpretation of alpha: since alpha > 0 (p <0.01), the counts are messy. using negative binomial is a good idea

In [9]:
from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm2, po_lm2, nb_m1])
sg.title('Number of Children')
sg.custom_columns(['OLS', 'Poisson', 'Neg. Binomial'], [1, 1, 1])

# Ending the cell on the object itself renders the table in the notebook
sg

## Test for Overdispersion

Was NB necessary? We test whether the NB's dispersion parameter (alpha) is significantly different from 0. If alpha = 0, the NB reduces to Poisson.

This is equivalent to R's `pscl::odTest()`.

In [33]:
# Overdispersion test: run a likelihood ratio test!
p_od  = 0.5 * chi2.sf(lr_od, 1) # looks up the p-value of alpha
print(f'Overdispersion LR statistic = {lr_od:.4f}')
print(f'p-value (one-sided) = {p_od:.6f}')
print('NB preferred over Poisson:', p_od < 0.05)
# # since p-value is < 0.01, we reject the null hypothesis. negative binomial model is better 

Overdispersion LR statistic = 56.2895
p-value (one-sided) = 0.000000
NB preferred over Poisson: True


# Marginal Effects

A marginal effect is the expected change in the number of children when an independent variable increases by one unit, ceteris paribus. Compute them the same way as for the logit models.

In [11]:
from marginaleffects import avg_comparisons, comparisons

## Average marginal effects of income

In [12]:
avg_comparisons(po_lm2, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001238,0.007496,0.165144,0.868891,0.202754,-0.013487,0.015963


a 10k increase of income is associated, on average across the sample, with an increase of 0.001238 in the number of children, holding all else constant. at p = 0.868, we do not reject the null hypothesis. there is no effect at the 0.05 threshold.

In [13]:
avg_comparisons(nb_m1, variables='income_10k')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.00077,0.009532,0.080801,0.93563,0.095991,-0.017952,0.019493


a 10k increase of income is associated, on average across the sample, with an increase of 0.00077 in the number of children, holding all else constant. at p = 0.0959, we do not reject the null hypotehsis. there is no effect

## Marginal effect at the mean of income

In [14]:
comparisons(po_lm2, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.001201,0.007274,0.165154,0.868883,0.202767,-0.013086,0.015488


a 10k increase in income is associated, for a typical average person, with an increase of 0.001201 number of children, holding all else constant. at p = 0.8688, we do not reject the null hypothesis. there is no effect at the 0.05 threshold

In [15]:
comparisons(nb_m1, variables='income_10k', newdata='mean')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""income_10k""","""+1""",0.000746,0.009229,0.080801,0.935629,0.095991,-0.017383,0.018874


a 10k increase in income is associated, for a typical average person, with an increase of 0.0000746 number of children, holding all else constant. at p = 0.935, we do not reject the null hypothesis. there is no effect at the 0.05 threshold

# Independent Investigation: Education and Number of Children

Investigate the relationship between **no. of children** (`biokids`) and **BA education** (`ba_binary`).

Do the following:

1. Include these independent variables: `ba_binary`, `age`, `income_10k`, `sex_woman`, and `race_bin`.
2. Build OLS, Poisson, and Negative Binomial models.
3. Produce a Stargazer table comparing the three models.
4. Compute marginal effects for each model.
5. Decide which model is best. Justify.
6. Answer: does a person with a BA have more or fewer children, and by how much? For the Poisson and negative binomial models, report a factor of change or a percentage change.

In [17]:
# OLS model 
ols_lm2 = smf.ols('biokids ~ ba_binary + age + income_10k + sex_woman + race_bin', data=df).fit()
print(ols_lm2.summary())

                            OLS Regression Results                            
Dep. Variable:                biokids   R-squared:                       0.118
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     12.29
Date:                Thu, 12 Nov 2026   Prob (F-statistic):           5.36e-13
Time:                        01:41:55   Log-Likelihood:                -1015.5
No. Observations:                 557   AIC:                             2045.
Df Residuals:                     550   BIC:                             2075.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.4837      0

In [18]:
# Poisson model 
po_lm2 = smf.poisson('biokids ~ ba_binary + age + income_10k + sex_woman + race_bin', data=df).fit()
print(po_lm2.summary())

Optimization terminated successfully.
         Current function value: 1.671346
         Iterations 5
                          Poisson Regression Results                          
Dep. Variable:                biokids   No. Observations:                  557
Model:                        Poisson   Df Residuals:                      550
Method:                           MLE   Df Model:                            6
Date:                Thu, 12 Nov 2026   Pseudo R-squ.:                 0.05449
Time:                        01:42:33   Log-Likelihood:                -930.94
converged:                       True   LL-Null:                       -984.59
Covariance Type:            nonrobust   LLR p-value:                 7.505e-21
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.3234      0.112     -2.885      0.004      -0.543      -0.104
sex_w

Interpretation: having a ba education is expected to cause a 0.3114 (p < 0.01) decrease in the log count of number of children, holding all else constant

In [ ]:
# negative binomial model 
nb_m2 = smf.negativebinomial('biokids ~ ba_binary + age + income_10k + sex_woman + race_bin', data=df).fit()
print(nb_m2.summary())

Optimization terminated successfully.
         Current function value: 1.637030
         Iterations: 28
         Function evaluations: 34
         Gradient evaluations: 34
                     NegativeBinomial Regression Results                      
Dep. Variable:                biokids   No. Observations:                  557
Model:               NegativeBinomial   Df Residuals:                      550
Method:                           MLE   Df Model:                            6
Date:                Thu, 12 Nov 2026   Pseudo R-squ.:                 0.03673
Time:                        01:43:06   Log-Likelihood:                -911.83
converged:                       True   LL-Null:                       -946.59
Covariance Type:            nonrobust   LLR p-value:                 5.111e-13
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             

Interpretation: having a ba education is expected to cause a 0.3301 decrease in the log count of number of children (p < 0.01), holding all else constant

In [34]:
# Overdispersion test
lr_od2 = 2 * (nb_m2.llf - po_lm2.llf)
p_od2  = 0.5 * chi2.sf(lr_od2, 1) # looks up the p-value of alpha
print(f'Overdispersion LR statistic = {lr_od2:.4f}')
print(f'p-value (one-sided) = {p_od2:.6f}')
print('NB preferred over Poisson:', p_od2 < 0.05)

Overdispersion LR statistic = 38.2281
p-value (one-sided) = 0.000000
NB preferred over Poisson: True


In [26]:
# compare models using staragazer 

from stargazer.stargazer import Stargazer

sg = Stargazer([ols_lm2, po_lm2, nb_m2])
sg.title('Sex, Race, Education, Age, Income')

sg.dependent_variable_name('Biokids')
sg.custom_columns(['Model 1', 'Model 2', 'Model 3'], [1, 1, 1])
sg.covariate_order(['ba_binary', 'age', 'income_10k', 'sex_woman[T.Woman]', 'race_bin[T.2.0]', 'Intercept'])
sg.rename_covariates({
'ba_binary': 'BA education', 
'age': 'Age',
'income_10k': 'Income',
'sex_woman[T.Woman]': 'Sex', 
'race_bin[T.2.0]': 'Race', 
'Intercept': 'Intercept'
})

sg

In [ ]:
# average marginal effects for OLS model
avg_comparisons(ols_lm2, variables='ba_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",-0.515433,0.145226,-3.54919,0.000419,11.219114,-0.800698,-0.230169


In [28]:
# average marginal effect for Poisson 
avg_comparisons(po_lm2, variables='ba_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",-0.485716,0.116509,-4.168926,0.000036,14.779926,-0.714573,-0.25686


In [29]:
# average marginal effect for NB
avg_comparisons(nb_m2, variables='ba_binary')

term,contrast,estimate,std_error,statistic,p_value,s_value,conf_low,conf_high
str,str,f64,f64,f64,f64,f64,f64,f64
"""ba_binary""","""1 - 0""",-0.515433,0.145226,-3.54919,0.000419,11.219114,-0.800698,-0.230169


In [35]:
# comparing aic values 
print(f'AIC ols_lm2: {ols_lm2.aic:.2f}')
print(f'AIC po_lm2: {po_lm2.aic:.2f}')
print(f'AIC nb_m2: {nb_m2.aic:.2f}')

AIC ols_lm2: 2044.94
AIC po_lm2: 1875.88
AIC nb_m2: 1839.65


the negative binomial model is preferred. firstly, it has the lowest AIC value among all 3 models. secondly, after running a likelihood ratio test, we can see that the data is quite spread out, and thus a negative binomial model is more appropriate to use than a poisson model. lastly, after running average marginal effect tests for each model, we can see that the p-value for the negative binomial model is < 0.001, so we reject the null hypotehsis that there education has no effect on the number of children

In [36]:
!jupyter nbconvert --to pdf week8_notebook_student.ipynb

[NbConvertApp] Converting notebook week8_notebook_student.ipynb to pdf
[NbConvertApp] Writing 72338 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 80346 bytes to week8_notebook_student.pdf
